# LeetCode 3903 — Smallest Stable Index I bằng Python

Link đề: https://leetcode.com/problems/smallest-stable-index-i/ · Độ khó: **Easy** · Chủ đề: Array, Prefix Maximum, Suffix Minimum

Skeleton LeetCode:

```python
class Solution:
    def firstStableIndex(self, nums: List[int], k: int) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi chỉ số `i`, **điểm bất ổn** = (giá trị lớn nhất của đoạn `nums[0..i]`) trừ (giá trị nhỏ nhất của đoạn `nums[i..n-1]`); `i` là **ổn định** nếu điểm `<= k`. Tìm chỉ số ổn định nhỏ nhất.
2. Nắm kỹ thuật **prefix maximum** (max tiền tố) và **suffix minimum** (min hậu tố), cùng cách tính chúng bằng một vòng lặp.
3. Biết `itertools.accumulate` với `max`/`min`, tạo mảng hậu tố bằng cách duyệt ngược và `heapq` với xóa lười (lazy deletion).
4. Viết 5 cách khác nhau và đối chiếu bằng test ngẫu nhiên có `random.seed`, kể cả giá trị lớn tới `10^9`.
5. Tránh lỗi: dùng max/min của cả mảng, dùng đoạn sai biên (loại trừ `i`), dùng `<` thay `<=`, trả điểm thay vì chỉ số.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n` và số nguyên `k`. Với mỗi chỉ số `i`, **điểm bất ổn** của `i` là

`max(nums[0..i]) - min(nums[i..n-1])`

(max lấy trên đoạn từ đầu mảng đến `i` **tính cả `i`**, min lấy trên đoạn từ `i` đến cuối mảng **tính cả `i`**). Chỉ số `i` là **ổn định** nếu điểm bất ổn của nó `<= k`. Trả về chỉ số ổn định **nhỏ nhất**, hoặc `-1` nếu không có.

| Ví dụ | nums | k | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[5,0,1,4]` | `3` | `3` | Điểm các chỉ số 0..3 là `5, 5, 4, 1`; chỉ số 3 là đầu tiên có điểm `<= 3` |
| 2 | `[3,2,1]` | `1` | `-1` | Mọi chỉ số có điểm `2 > 1` |
| 3 | `[0]` | `0` | `0` | Điểm `0 - 0 = 0 <= 0` |

Constraints: `1 <= nums.length <= 100`; `0 <= nums[i] <= 10^9`; `0 <= k <= 10^9`.

In [1]:
def scores(nums):
    n = len(nums)
    return [max(nums[:i + 1]) - min(nums[i:]) for i in range(n)]

for nums, k in [([5, 0, 1, 4], 3), ([3, 2, 1], 1), ([0], 0)]:
    print(nums, "k =", k, "điểm =", scores(nums))

[5, 0, 1, 4] k = 3 điểm = [5, 5, 4, 1]
[3, 2, 1] k = 1 điểm = [2, 2, 2]
[0] k = 0 điểm = [0]


## 3. Kiến thức nền

### 3.1. Cắt lát và `max`/`min` của một đoạn

`nums[:i + 1]` là đoạn `0..i` (gồm `i`), `nums[i:]` là đoạn `i..n-1` (gồm `i`). `max(...)`, `min(...)` tính trên đoạn đó, mỗi lần `O(độ dài)`.

In [2]:
nums = [5, 0, 1, 4]
i = 2
print(nums[:i + 1], max(nums[:i + 1]), nums[i:], min(nums[i:]))

[5, 0, 1] 5 [1, 4] 1


### 3.2. Prefix maximum (max tiền tố)

`pre[i] = max(nums[0..i])`. Tính được trong một vòng: `pre[i] = max(pre[i-1], nums[i])`. Dãy `pre` **không giảm**.

In [3]:
nums = [5, 0, 1, 4]
pre = []
cur = float('-inf')
for x in nums:
    cur = max(cur, x)
    pre.append(cur)
print(pre)

[5, 5, 5, 5]


### 3.3. Suffix minimum (min hậu tố)

`suf[i] = min(nums[i..n-1])`. Tính bằng cách duyệt **ngược**: `suf[i] = min(suf[i+1], nums[i])`. Dãy `suf` **không giảm** theo `i` (đi về phải, đoạn ngắn lại nên min chỉ có thể tăng).

In [4]:
nums = [5, 0, 1, 4]
n = len(nums)
suf = [0] * n
cur = float('inf')
for i in range(n - 1, -1, -1):
    cur = min(cur, nums[i])
    suf[i] = cur
print(suf)

[0, 0, 1, 4]


### 3.4. `itertools.accumulate`

`accumulate(iterable, func)` cho dãy kết quả tích lũy: `accumulate(nums, max)` là prefix max. Với suffix min: đảo, tích lũy, đảo lại.

In [5]:
from itertools import accumulate
nums = [5, 0, 1, 4]
print(list(accumulate(nums, max)))
print(list(accumulate(reversed(nums), min))[::-1])

[5, 5, 5, 5]
[0, 0, 1, 4]


### 3.5. `heapq` và xóa lười (lazy deletion)

`heapq` là hàng đợi ưu tiên nhỏ nhất ở đầu. Muốn lấy **min của `nums[i:]`** khi `i` tăng dần, ta bỏ các phần tử có chỉ số `< i` khỏi đỉnh heap **khi chúng nổi lên** (không xóa ngay).

In [6]:
import heapq
nums = [5, 0, 1, 4]
heap = [(x, idx) for idx, x in enumerate(nums)]
heapq.heapify(heap)
for i in range(len(nums)):
    while heap[0][1] < i:
        heapq.heappop(heap)
    print(f"i={i}: min(nums[{i}:]) = {heap[0][0]}")

i=0: min(nums[0:]) = 0
i=1: min(nums[1:]) = 0
i=2: min(nums[2:]) = 1
i=3: min(nums[3:]) = 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Điểm bất ổn của `i` là "chênh lệch" giữa **đỉnh cao nhất đã thấy từ đầu đến `i`** và **đáy thấp nhất từ `i` đến cuối**. Bạn có thể hình dung `nums` là dãy các cột: đứng ở cột `i`, nhìn lại phía trái để tìm cột cao nhất, nhìn về phía phải để tìm cột thấp nhất.

Tính tay `[5, 0, 1, 4]`, `k = 3`:

| i | `max(nums[0..i])` | `min(nums[i..])` | điểm | `<= 3`? |
|---|---|---|---|---|
| 0 | 5 | 0 | 5 | không |
| 1 | 5 | 0 | 5 | không |
| 2 | 5 | 1 | 4 | không |
| 3 | 5 | 4 | 1 | **có** |

Chỉ số ổn định nhỏ nhất là `3`.

Mỗi giá trị trên bảng đều dùng lại phần đã tính: `max` tiền tố chỉ cần cập nhật với phần tử mới; `min` hậu tố tính được một lần từ phải sang trái. Nhờ vậy không cần quét lại đoạn mỗi lần.

In [7]:
nums, k = [5, 0, 1, 4], 3
n = len(nums)
pre, cur = [], float('-inf')
for x in nums:
    cur = max(cur, x); pre.append(cur)
suf, cur = [0] * n, float('inf')
for i in range(n - 1, -1, -1):
    cur = min(cur, nums[i]); suf[i] = cur
print("pre =", pre, "suf =", suf, "điểm =", [pre[i] - suf[i] for i in range(n)])

pre = [5, 5, 5, 5] suf = [0, 0, 1, 4] điểm = [5, 5, 4, 1]


## 5. Cách 1 — Brute Force / baseline

Với mỗi `i`, tính `max(nums[:i + 1]) - min(nums[i:])` rồi so với `k`; trả `i` đầu tiên thỏa.

Nút thắt: mỗi `i` quét lại đoạn tiền tố và hậu tố nên tổng `O(n²)`. Với `n <= 100` vẫn nhanh nhưng lãng phí, vì `max` tiền tố của `i` là `max` tiền tố của `i - 1` cộng thêm đúng một phần tử.

In [8]:
class SolutionBrute:
    def firstStableIndex(self, nums, k):
        n = len(nums)
        for i in range(n):
            if max(nums[:i + 1]) - min(nums[i:]) <= k:
                return i
        return -1

print(SolutionBrute().firstStableIndex([5, 0, 1, 4], 3), SolutionBrute().firstStableIndex([3, 2, 1], 1), SolutionBrute().firstStableIndex([0], 0))

3 -1 0


**Độ phức tạp:** Time `O(n²)`, Space `O(n)` cho các lát cắt.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách thành hai mảng độc lập, mỗi mảng tính bằng một lượt:

- `pre[i] = max(nums[0..i])` bằng một lượt trái -> phải;
- `suf[i] = min(nums[i..n-1])` bằng một lượt phải -> trái.

Khi đó điểm bất ổn của `i` là `pre[i] - suf[i]`, tra cứu `O(1)`. Chỉ số ổn định đầu tiên là `i` nhỏ nhất với `pre[i] - suf[i] <= k`.

**Invariant:** trong lượt duyệt trái -> phải, biến `cur_max` luôn bằng `max(nums[0..i])` sau khi cập nhật bằng `nums[i]`; `suf` được dựng xong **trước** nên `suf[i]` luôn sẵn dùng. Cách tiết kiệm bộ nhớ: chỉ lưu `suf` (mảng) còn prefix max là một **biến chạy**.

Lưu ý: điểm bất ổn **không đơn điệu** (có thể tăng rồi giảm, như ví dụ `5,5,4,1`; nhưng cũng có thể giảm rồi tăng), nên không dùng được tìm kiếm nhị phân; phải duyệt tuyến tính và dừng ở chỉ số ổn định đầu tiên.

## 7. Các cách tiếp theo

### Cách 2 — Hai mảng `pre` và `suf`

Dựng hai mảng đầy đủ rồi duyệt `i` tăng dần.

In [9]:
class SolutionTwoArrays:
    def firstStableIndex(self, nums, k):
        n = len(nums)
        pre, suf = [0] * n, [0] * n
        pre[0] = nums[0]
        for i in range(1, n):
            pre[i] = max(pre[i - 1], nums[i])
        suf[n - 1] = nums[n - 1]
        for i in range(n - 2, -1, -1):
            suf[i] = min(suf[i + 1], nums[i])
        for i in range(n):
            if pre[i] - suf[i] <= k:
                return i
        return -1

print(SolutionTwoArrays().firstStableIndex([5, 0, 1, 4], 3), SolutionTwoArrays().firstStableIndex([3, 2, 1], 1), SolutionTwoArrays().firstStableIndex([0], 0))

3 -1 0


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 3 — Chỉ mảng `suf` + biến chạy `cur_max`

Giảm bộ nhớ một nửa: prefix max là biến `cur_max` cập nhật ngay trong lượt kiểm tra.

In [10]:
class SolutionSufRunning:
    def firstStableIndex(self, nums, k):
        n = len(nums)
        suf = [0] * n
        suf[-1] = nums[-1]
        for i in range(n - 2, -1, -1):
            suf[i] = min(suf[i + 1], nums[i])
        cur_max = nums[0]
        for i in range(n):
            cur_max = max(cur_max, nums[i])
            if cur_max - suf[i] <= k:
                return i
        return -1

print(SolutionSufRunning().firstStableIndex([5, 0, 1, 4], 3), SolutionSufRunning().firstStableIndex([3, 2, 1], 1), SolutionSufRunning().firstStableIndex([0], 0))

3 -1 0


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (một mảng).

### Cách 4 — `accumulate`

Prefix max bằng `accumulate(nums, max)`, suffix min bằng `accumulate(reversed(nums), min)` đảo lại; rồi `zip` và `next`.

In [11]:
from itertools import accumulate

class SolutionAccumulate:
    def firstStableIndex(self, nums, k):
        pre = accumulate(nums, max)
        suf = list(accumulate(reversed(nums), min))[::-1]
        return next((i for i, (p, s) in enumerate(zip(pre, suf)) if p - s <= k), -1)

print(SolutionAccumulate().firstStableIndex([5, 0, 1, 4], 3), SolutionAccumulate().firstStableIndex([3, 2, 1], 1), SolutionAccumulate().firstStableIndex([0], 0))

3 -1 0


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 5 — `heapq` với xóa lười để lấy `min(nums[i:])`

Không dựng mảng hậu tố; thay vào đó, duy trì min-heap của `(giá_trị, chỉ_số)`. Khi xét `i`, loại các phần tử có chỉ số `< i` ở đỉnh heap; đỉnh còn lại chính là `min(nums[i:])`. Hữu ích để làm quen xóa lười, nhưng chậm hơn (`O(n log n)`).

In [12]:
import heapq

class SolutionHeap:
    def firstStableIndex(self, nums, k):
        heap = [(x, idx) for idx, x in enumerate(nums)]
        heapq.heapify(heap)
        cur_max = nums[0]
        for i, x in enumerate(nums):
            cur_max = max(cur_max, x)
            while heap[0][1] < i:
                heapq.heappop(heap)
            if cur_max - heap[0][0] <= k:
                return i
        return -1

print(SolutionHeap().firstStableIndex([5, 0, 1, 4], 3), SolutionHeap().firstStableIndex([3, 2, 1], 1), SolutionHeap().firstStableIndex([0], 0))

3 -1 0


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 với `nums = [5, 0, 1, 4]`, `k = 3`. Mảng hậu tố `suf = [0, 0, 1, 4]`.

| i | nums[i] | cur_max | suf[i] | điểm | `<= 3`? |
|---|---|---|---|---|---|
| 0 | 5 | 5 | 0 | 5 | không |
| 1 | 0 | 5 | 0 | 5 | không |
| 2 | 1 | 5 | 1 | 4 | không |
| 3 | 4 | 5 | 4 | 1 | có -> trả 3 |

In [13]:
def trace_suf_running(nums, k):
    n = len(nums)
    suf = [0] * n
    suf[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        suf[i] = min(suf[i + 1], nums[i])
    print("suf =", suf)
    cur_max = nums[0]
    for i in range(n):
        cur_max = max(cur_max, nums[i])
        score = cur_max - suf[i]
        print(f"i={i} nums[i]={nums[i]} cur_max={cur_max} suf[i]={suf[i]} điểm={score} <= {k}? {score <= k}")
        if score <= k:
            return i
    return -1

assert trace_suf_running([5, 0, 1, 4], 3) == 3
assert trace_suf_running([3, 2, 1], 1) == -1
assert trace_suf_running([0], 0) == 0

suf = [0, 0, 1, 4]
i=0 nums[i]=5 cur_max=5 suf[i]=0 điểm=5 <= 3? False
i=1 nums[i]=0 cur_max=5 suf[i]=0 điểm=5 <= 3? False
i=2 nums[i]=1 cur_max=5 suf[i]=1 điểm=4 <= 3? False
i=3 nums[i]=4 cur_max=5 suf[i]=4 điểm=1 <= 3? True
suf = [1, 1, 1]
i=0 nums[i]=3 cur_max=3 suf[i]=1 điểm=2 <= 1? False
i=1 nums[i]=2 cur_max=3 suf[i]=1 điểm=2 <= 1? False
i=2 nums[i]=1 cur_max=3 suf[i]=1 điểm=2 <= 1? False
suf = [0]
i=0 nums[i]=0 cur_max=0 suf[i]=0 điểm=0 <= 0? True


In [14]:
def trace_heap(nums, k):
    heap = [(x, idx) for idx, x in enumerate(nums)]
    heapq.heapify(heap)
    cur_max = nums[0]
    for i, x in enumerate(nums):
        cur_max = max(cur_max, x)
        popped = []
        while heap[0][1] < i:
            popped.append(heapq.heappop(heap))
        print(f"i={i}: loại {popped}, đỉnh heap {heap[0]}, điểm {cur_max} - {heap[0][0]} = {cur_max - heap[0][0]}")
        if cur_max - heap[0][0] <= k:
            return i
    return -1

assert trace_heap([5, 0, 1, 4], 3) == 3

i=0: loại [], đỉnh heap (0, 1), điểm 5 - 0 = 5
i=1: loại [], đỉnh heap (0, 1), điểm 5 - 0 = 5
i=2: loại [(0, 1)], đỉnh heap (1, 2), điểm 5 - 1 = 4
i=3: loại [(1, 2)], đỉnh heap (4, 3), điểm 5 - 4 = 1


## 9. Code nộp LeetCode

Dùng Cách 3: `O(n)` thời gian, một mảng hậu tố và một biến max chạy.

In [15]:
from typing import List, Optional


class Solution:
    def firstStableIndex(self, nums: List[int], k: int) -> int:
        n = len(nums)
        suf = [0] * n                       # suf[i] = min(nums[i..n-1])
        suf[-1] = nums[-1]
        for i in range(n - 2, -1, -1):
            suf[i] = min(suf[i + 1], nums[i])
        cur_max = nums[0]                   # max(nums[0..i])
        for i in range(n):
            cur_max = max(cur_max, nums[i])
            if cur_max - suf[i] <= k:
                return i
        return -1


print(Solution().firstStableIndex([5, 0, 1, 4], 3), Solution().firstStableIndex([3, 2, 1], 1), Solution().firstStableIndex([0], 0))

3 -1 0


## 10. Vì sao thuật toán đúng

1. **Prefix max đúng.** `cur_max` trước khi xử lý `i` bằng `max(nums[0..i-1])`; sau khi `max` với `nums[i]` thì bằng `max(nums[0..i])`. Ban đầu `cur_max = nums[0]` đúng cho `i = 0`.
2. **Suffix min đúng.** `suf[n-1] = nums[n-1]`; với `i < n-1`, `min(nums[i..n-1]) = min(nums[i], min(nums[i+1..n-1]))`, đúng công thức đệ quy.
3. **Điểm bất ổn.** `cur_max - suf[i]` đúng bằng định nghĩa `max(nums[0..i]) - min(nums[i..n-1])`. Cả hai đoạn **đều chứa `i`**, khớp đề.
4. **Chỉ số nhỏ nhất.** Duyệt `i` tăng dần và trả về ngay khi điểm `<= k`, nên chỉ số trả về là nhỏ nhất.
5. **Không tồn tại.** Nếu không có `i` nào, vòng lặp kết thúc và trả `-1`.
6. **Giá trị lớn.** Phép trừ nguyên của Python không tràn số, nên `10^9` không gây vấn đề (ở ngôn ngữ khác có thể cần kiểu 64-bit).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- Cách 1: `O(n²)` thời gian, `O(n)` bộ nhớ cho lát cắt.
- Cách 2: `O(n)` thời gian, `O(n)` bộ nhớ (hai mảng).
- Cách 3, 4: `O(n)` thời gian, `O(n)` bộ nhớ (một mảng, hoặc `accumulate`).
- Cách 5: `O(n log n)` thời gian (dựng heap và pop), `O(n)` bộ nhớ.

Best case: `i = 0` đã ổn định (nhưng mảng `suf` vẫn dựng `O(n)`). Worst case: không có chỉ số ổn định, duyệt hết.

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên (độ dài 1, mảng tăng/giảm, `k = 0`, giá trị `10^9`) và 400 input ngẫu nhiên đúng constraints.

In [16]:
import random

solutions = [SolutionBrute(), SolutionTwoArrays(), SolutionSufRunning(), SolutionAccumulate(), SolutionHeap(), Solution()]

fixed = [
    (([5, 0, 1, 4], 3), 3),
    (([3, 2, 1], 1), -1),
    (([0], 0), 0),
    (([1, 2, 3], 0), 0),                # mảng tăng: điểm mọi chỉ số bằng nums[i] - nums[i] = 0
    (([3, 2, 1], 2), 0),                # điểm chỉ số 0: 3 - 1 = 2
    (([10**9, 0], 10**9), 0),
    (([10**9, 0], 10**9 - 1), -1),
    (([5, 5, 5], 0), 0),
    (([2, 9, 1], 5), 0),                # chỉ số 0: max(2) - min(2,9,1) = 1 <= 5
]
for (nums, k), expected in fixed:
    for s in solutions:
        assert s.firstStableIndex(list(nums), k) == expected, (type(s).__name__, nums, k)

random.seed(39030)
for _ in range(400):
    n = random.randint(1, 100)
    hi = random.choice([3, 20, 10**9])
    nums = [random.randint(0, hi) for _ in range(n)]
    k = random.randint(0, hi)
    outs = [s.firstStableIndex(list(nums), k) for s in solutions]
    assert all(o == outs[0] for o in outs), (nums, k, outs)
    if outs[0] != -1:
        i = outs[0]
        assert max(nums[:i + 1]) - min(nums[i:]) <= k
        assert all(max(nums[:j + 1]) - min(nums[j:]) > k for j in range(i))
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) `k` rất lớn (`>= max - min` của cả mảng) thì đáp án là `0`. (2) Điểm bất ổn của chỉ số cuối là `max(nums) - nums[-1]`, của chỉ số đầu là `nums[0] - min(nums)`. (3) Nếu tăng `k` thì đáp án (nếu có) không tăng.

In [17]:
random.seed(3903)
for _ in range(300):
    n = random.randint(1, 60)
    nums = [random.randint(0, 10**9) for _ in range(n)]
    assert Solution().firstStableIndex(nums, max(nums) - min(nums)) == 0
    k = random.randint(0, 10**9)
    a = Solution().firstStableIndex(nums, k)
    b = Solution().firstStableIndex(nums, k + random.randint(0, 10**8))
    if a != -1:
        assert b != -1 and b <= a
    # kiểm tra công thức hai đầu
    last_score = max(nums) - nums[-1]
    assert (Solution().firstStableIndex(nums, last_score) != -1)
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng max và min của **cả mảng**

Điểm bất ổn dùng **tiền tố** cho max và **hậu tố** cho min, không phải toàn mảng. Với `[5,0,1,4]`, `k = 3`, max - min toàn mảng là `5` nên không bao giờ `<= 3`.

In [18]:
def wrong_global(nums, k):
    d = max(nums) - min(nums)                                      # SAI: dùng cả mảng
    return 0 if d <= k else -1

print(wrong_global([5, 0, 1, 4], 3), "!=", Solution().firstStableIndex([5, 0, 1, 4], 3))
assert wrong_global([5, 0, 1, 4], 3) == -1 and Solution().firstStableIndex([5, 0, 1, 4], 3) == 3

-1 != 3


### Lỗi 2: lệch biên đoạn (loại trừ chỉ số `i`)

Dùng `max(nums[:i])` (không gồm `i`) hoặc `min(nums[i + 1:])` (không gồm `i`) làm đoạn rỗng ở biên và cho kết quả khác. Cả hai đoạn của đề **đều chứa `i`**.

In [19]:
def wrong_exclusive(nums, k):
    n = len(nums)
    for i in range(1, n - 1):                                      # SAI: bỏ i = 0 và i = n - 1 vì đoạn rỗng
        if max(nums[:i]) - min(nums[i + 1:]) <= k:
            return i
    return -1

print(wrong_exclusive([0], 0), wrong_exclusive([5, 0, 1, 4], 3), "!=", Solution().firstStableIndex([0], 0), Solution().firstStableIndex([5, 0, 1, 4], 3))
assert wrong_exclusive([0], 0) == -1 and Solution().firstStableIndex([0], 0) == 0
assert wrong_exclusive([5, 0, 1, 4], 3) == 2 and Solution().firstStableIndex([5, 0, 1, 4], 3) == 3

-1 2 != 0 3


### Lỗi 3: dùng `<` thay vì `<=`

Đề nói ổn định khi điểm **nhỏ hơn hoặc bằng** `k`. Với `[0]`, `k = 0` điểm là `0`; `0 < 0` sai nên trả `-1`.

In [20]:
def wrong_strict(nums, k):
    n = len(nums)
    for i in range(n):
        if max(nums[:i + 1]) - min(nums[i:]) < k:                  # SAI: phải <=
            return i
    return -1

print(wrong_strict([0], 0), "!=", Solution().firstStableIndex([0], 0))
assert wrong_strict([0], 0) == -1 and Solution().firstStableIndex([0], 0) == 0
assert wrong_strict([5, 0, 1, 4], 1) == -1 and Solution().firstStableIndex([5, 0, 1, 4], 1) == 3

-1 != 0


### Lỗi 4: trả về chỉ số **cuối cùng** thay vì đầu tiên (hoặc điểm thay vì chỉ số)

Đề hỏi chỉ số **nhỏ nhất**. Dùng `if ...: ans = i` mà không `return` sẽ nhận chỉ số lớn nhất. Trả điểm thay vì chỉ số cũng sai.

In [21]:
def wrong_last(nums, k):
    ans = -1
    for i in range(len(nums)):
        if max(nums[:i + 1]) - min(nums[i:]) <= k:
            ans = i                                                # SAI: không dừng ở chỉ số đầu tiên
    return ans

def wrong_return_score(nums, k):
    for i in range(len(nums)):
        s = max(nums[:i + 1]) - min(nums[i:])
        if s <= k:
            return s                                               # SAI: trả điểm
    return -1

print(wrong_last([1, 2, 3], 5), wrong_return_score([5, 0, 1, 4], 3), "!=", Solution().firstStableIndex([1, 2, 3], 5), Solution().firstStableIndex([5, 0, 1, 4], 3))
assert wrong_last([1, 2, 3], 5) == 2 and Solution().firstStableIndex([1, 2, 3], 5) == 0
assert wrong_return_score([5, 0, 1, 4], 3) == 1 and Solution().firstStableIndex([5, 0, 1, 4], 3) == 3

2 1 != 0 3


### Lỗi 5: tìm kiếm nhị phân vì tưởng điểm đơn điệu

Điểm bất ổn không đơn điệu theo `i`. Nhị phân có thể bỏ sót chỉ số ổn định nhỏ nhất.

In [22]:
def wrong_binary_search(nums, k):
    n = len(nums)
    score = lambda i: max(nums[:i + 1]) - min(nums[i:])
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if score(mid) <= k:
            hi = mid
        else:
            lo = mid + 1
    return lo if score(lo) <= k else -1                             # SAI: giả sử đơn điệu

nums, k = [0, 9, 0, 9], 9
print([max(nums[:i + 1]) - min(nums[i:]) for i in range(4)])
print(wrong_binary_search(nums, k), "vs", Solution().firstStableIndex(nums, k))
random.seed(1)
cnt = 0
for _ in range(500):
    a = [random.randint(0, 9) for _ in range(random.randint(1, 8))]
    b = random.randint(0, 9)
    cnt += wrong_binary_search(a, b) != Solution().firstStableIndex(a, b)
print("số input mà nhị phân sai (trên 500 mẫu nhỏ):", cnt)
assert cnt > 0

[0, 9, 9, 0]
0 vs 0
số input mà nhị phân sai (trên 500 mẫu nhỏ): 79


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "max của tiền tố" và "min của hậu tố" -> **prefix max / suffix min**: tính một lượt, tra cứu `O(1)`.
- Bài có "chỉ số nhỏ nhất thỏa điều kiện" -> duyệt tăng dần và **trả ngay** khi thỏa.
- Một mảng + một biến chạy thường đủ: mảng cho phía phải (tính trước), biến cho phía trái (cập nhật dần).
- Đừng suy diễn tính đơn điệu khi chưa chứng minh; test nhị phân với input ngẫu nhiên nhỏ để kiểm chứng.
- Đọc kỹ biên đoạn: `0..i` và `i..n-1` đều **gồm** `i`.

## 15. Biến thể của bài

1. **Smallest Stable Index II** (bài cùng cặp): ràng buộc lớn hơn/biến thể tương tự.
2. **Trapping Rain Water** (42): dùng prefix max và suffix max.
3. **Product of Array Except Self** (238) và **Best Time to Buy and Sell Stock** (121): tiền tố/hậu tố.
4. **Partition Array Into Disjoint Intervals** (915): max của tiền tố so với min của hậu tố.
5. **Find the Pivot Integer / Find Pivot Index** (724): so sánh hai phía.
6. Biến thể: tìm chỉ số ổn định **lớn nhất**, đếm **số lượng** chỉ số ổn định, hoặc điểm bất ổn nhỏ nhất.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | `max`/`min` lát cắt mỗi `i` | `O(n²)` | `O(n)` | Oracle |
| 2. Hai mảng | `pre`, `suf` đầy đủ | `O(n)` | `O(n)` | Dễ hình dung |
| 3. `suf` + biến chạy | `cur_max` cập nhật dần | `O(n)` | `O(n)` | Nộp bài |
| 4. `accumulate` | `accumulate(nums, max)` | `O(n)` | `O(n)` | Ngắn gọn |
| 5. Heap lười | min hậu tố qua heap | `O(n log n)` | `O(n)` | Luyện xóa lười |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().firstStableIndex([2, 1, 4], 1)` và `Solution().firstStableIndex([2, 1, 4], 0)` ra gì?

**Bài 2 (tính tay):** với `nums = [4, 2, 6, 1]`, tính điểm bất ổn của từng chỉ số.

**Bài 3 (điền TODO):** hoàn thành `all_scores(nums)` trả về **danh sách điểm** của mọi chỉ số (bằng hai mảng `pre`, `suf`).

**Bài 4 (tự viết lại):** viết `count_stable(nums, k)` đếm số chỉ số ổn định.

**Bài 5 (mini challenge):** viết `min_score_index(nums)` trả về chỉ số có điểm bất ổn **nhỏ nhất** (hòa thì chỉ số nhỏ nhất).

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().firstStableIndex([2, 1, 4], 1), Solution().firstStableIndex([2, 1, 4], 0))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def all_scores_todo(nums):
    n = len(nums)
    scores = []
    # TODO: dựng pre (max tiền tố) và suf (min hậu tố), rồi scores[i] = pre[i] - suf[i]
    return scores

0 2


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: tính điểm từng chỉ số cho `[2,1,4]`: chỉ số 0 có max `2`, min toàn mảng `1`. Bài 2: tương tự bảng ở mục 4.

**Mức 2 (vừa):** Bài 3: dùng hai vòng lặp như Cách 2. Bài 4: giống Cách 3 nhưng đếm thay vì `return`. Bài 5: tính mọi điểm rồi dùng `min` với khóa `(điểm, chỉ số)`.

**Mức 3 (gần đáp án):** Bài 1: điểm `[1, 1, 0]` nên `0` và `2`. Bài 2: điểm `[3, 3, 5, 5]` (`i=0`: `4 - 1`; `i=1`: `4 - 1`; `i=2`: `6 - 1`; `i=3`: `6 - 1`).

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().firstStableIndex([2, 1, 4], 1) == 0 and Solution().firstStableIndex([2, 1, 4], 0) == 2
assert scores([4, 2, 6, 1]) == [3, 3, 5, 5]

# Bài 3
def all_scores(nums):
    n = len(nums)
    pre, suf = [0] * n, [0] * n
    pre[0] = nums[0]
    for i in range(1, n):
        pre[i] = max(pre[i - 1], nums[i])
    suf[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        suf[i] = min(suf[i + 1], nums[i])
    return [pre[i] - suf[i] for i in range(n)]
assert all_scores([5, 0, 1, 4]) == [5, 5, 4, 1] and all_scores([0]) == [0] and all_scores([3, 2, 1]) == [2, 2, 2]

# Bài 4
def count_stable(nums, k):
    return sum(1 for s in all_scores(nums) if s <= k)
assert count_stable([5, 0, 1, 4], 3) == 1 and count_stable([5, 0, 1, 4], 5) == 4 and count_stable([3, 2, 1], 1) == 0

# Bài 5
def min_score_index(nums):
    sc = all_scores(nums)
    return min(range(len(nums)), key=lambda i: (sc[i], i))
assert min_score_index([5, 0, 1, 4]) == 3 and min_score_index([7]) == 0 and min_score_index([1, 2, 3]) == 0
random.seed(11)
for _ in range(100):
    a = [random.randint(0, 20) for _ in range(random.randint(1, 15))]
    assert all_scores(a) == scores(a)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tiền tố × hậu tố":**

```python
n = len(nums)
suf = [0] * n
suf[-1] = nums[-1]
for i in range(n - 2, -1, -1):
    suf[i] = min(suf[i + 1], nums[i])        # min hậu tố
cur_max = nums[0]
for i in range(n):
    cur_max = max(cur_max, nums[i])          # max tiền tố
    if cur_max - suf[i] <= k:
        return i
return -1
```

**Checklist trước khi submit:**

- [ ] Max lấy trên `nums[0..i]` và min lấy trên `nums[i..n-1]`, cả hai **gồm `i`**?
- [ ] Dùng `<= k` (không phải `< k`)?
- [ ] Trả về chỉ số **đầu tiên** (nhỏ nhất) thỏa, không phải chỉ số cuối hay điểm?
- [ ] Không giả sử điểm bất ổn đơn điệu (không dùng nhị phân)?
- [ ] Đã thử `[0]`, `[5,0,1,4]` với `k = 3`, `[3,2,1]` với `k = 1`?
- [ ] Chữ ký đúng `firstStableIndex(self, nums: List[int], k: int) -> int`?